In [1]:
import pandas as pd
from scipy import stats
from scipy.stats import kruskal
import statsmodels.api as sm
from statsmodels.formula.api import ols
import seaborn as sns
import matplotlib.pyplot as plt

In [12]:
curRoot = 'C'  # 'C' or 'D'

# Load shape measures
shape_path = rf'{curRoot}:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Isomap\isomapCmdsCSSylk10d3distmin_keepOut.txt'      # Region CSSyl
#shape_path = rf'{curRoot}:\A_joyCODE\DEV_LIN\ampExt\CSpreCS\Isomap\isomapCmdsCSpreCSk10d3distmin_keepOut.txt' # Region CSpreCS
shapeU_1_path = rf'{curRoot}:\A_joyCODE\DEV_LIN\Shape_study\CSSyl\AmpExt_Umap\dim1_neig5_minDist0.2.txt'
shapeU_2_path = rf'{curRoot}:\A_joyCODE\DEV_LIN\Shape_study\CSSyl\AmpExt_Umap\dim1_neig30_minDist0.2.txt'
shapeU_3_path = rf'{curRoot}:\A_joyCODE\DEV_LIN\Shape_study\CSSyl\AmpExt_Umap\dim2_neig5_minDist0.2.txt'
shapeU_4_path = rf'{curRoot}:\A_joyCODE\DEV_LIN\Shape_study\CSSyl\AmpExt_Umap\dim2_neig30_minDist0.2.txt'
#shapeU_4_path = r'D:\A_joyCODE\DEV_LIN\Shape_study\CSSyl\AmpExt_Umap\dim3_neig5_minDist0.2.txt'
try:
    shape = pd.read_csv(shape_path, index_col=0, header=0)
    print(shape.head())
    shapeU_1 = pd.read_csv(shapeU_1_path, index_col=0, header=0)
    shapeU_2 = pd.read_csv(shapeU_2_path, index_col=0, header=0)
    shapeU_3 = pd.read_csv(shapeU_3_path, index_col=0, header=0)
    shapeU_4 = pd.read_csv(shapeU_4_path, index_col=0, header=0)

    shapeU_1.rename(columns={'UMAP1': 'UMAP1_U1'}, inplace=True)
    shapeU_2.rename(columns={'UMAP1': 'UMAP1_U2'}, inplace=True)
    shapeU_3.rename(columns={'UMAP1': 'UMAP1_U3', 'UMAP2': 'UMAP2_U3'}, inplace=True)
    shapeU_4.rename(columns={'UMAP1': 'UMAP1_U4', 'UMAP2': 'UMAP2_U4'}, inplace=True)
    #shapeU_4.rename(columns={'UMAP1': 'UMAP1_U4', 'UMAP2': 'UMAP2_U4','UMAP3': 'UMAP3_U4'}, inplace=True)
except FileNotFoundError as e:
    print(f"Error: {e}")

#shape_joined = shape.join(shapeU)
shape_joined = shape.join([shapeU_1, shapeU_2, shapeU_3, shapeU_4])
shape = shape_joined
#print(shape.head())

                     1         2         3
subjName                                  
LPC24_struct -3.129764 -0.129110 -1.895557
LPC05_struct -3.857513 -3.494893  1.386982
LPA30_struct  4.102761  1.443144 -0.599581
LPC21_struct -1.398569 -1.033002  0.397889
LPA32_struct -3.843092  2.432006  1.754709


In [14]:
# Modify the subject names of shape measures to agree with that of the INFO files
# Remove all postfix
def remove_postfix(index_name):
    return index_name.split('_')[0]
shape['index_modified'] = shape.index.map(remove_postfix)
shape['index_ori'] = shape.index
shape_indexed = shape.set_index('index_modified') 
#print("\nDataFrame with Modified Index Column:")
#print(shape_indexed)

In [18]:
# Load anatomical and functional info 
amp_info_path = rf'{curRoot}:\B_projWIP\proj_amputee\amputee_INFO_result\allinfoAmputeeJoy.csv'
function_path = rf'{curRoot}:\B_projWIP\proj_amputee\amputee_INFO_result\Functional_plasticity.csv'
try:
    shape_info = pd.read_csv(amp_info_path, index_col=0, header=0)
#    print("***********  Shape INFO  ************")
#    print(shape_info.head())
    function_info = pd.read_csv(function_path,index_col=0, header=0)
#    print("***********  Activation congenital INFO  ************")
#    print(function_info.head())
except FileNotFoundError as e:
    print(f"Error: {e}")
# set subjID as index for function_info
function_info_reset = function_info.reset_index()
function_info_reset.set_index('subjID', inplace=True)
function_info = function_info_reset
#print(function_info.head())

In [458]:
print(len(shape_info))
print(len(function_info))

65
24


In [20]:
# Double shape_info to account for the two hemisphere
shape_info_L = shape_info.copy()
shape_info_flip_R = shape_info.copy()
# Append 'L' to the index of the first copy
shape_info_L.index = 'L' + shape_info_L.index
# Append 'flip_R' to the index of the second copy
shape_info_flip_R.index = 'flip-R' + shape_info_flip_R.index
# Compose the new info df with the new index
shape_info_LR_combined = pd.concat([shape_info_L, shape_info_flip_R])
#print(shape_info_LR_combined)

# Do the same for function_info
function_info_L = function_info.copy()
function_info_flip_R = function_info.copy()
function_info_L.index = 'L' + function_info_L.index
function_info_flip_R.index = 'flip-R' + function_info_flip_R.index
function_info_LR_combined = pd.concat([function_info_L, function_info_flip_R])
#print(function_info_LR_combined.columns)

In [22]:
# Join INFO with shape measures
desired_columns = ['index_ori','1', '2', '3','UMAP1_U1','UMAP1_U2','UMAP1_U3','UMAP2_U3','UMAP1_U4','UMAP2_U4'] 
# IF needing additional columns
#desired_columns = ['1', '2', '3','UMAP1_U1','UMAP1_U2','UMAP1_U3','UMAP2_U3','UMAP1_U4','UMAP2_U4','UMAP3_U4'] 
#joined_info_shape = pd.concat([shape_info_LR_combined, shape_indexed[desired_columns]], axis=1)


# Inner join on index, selecting desired columns from df2
joined_info_shape = shape_info_LR_combined.merge(shape_indexed[desired_columns], how='inner', left_index=True, right_index=True)
#print(joined_info_shape.columns)
#print(joined_info_shape.index)

In [26]:
# Adding additional columns: side, contraAmpSide and study
# Shape measure rename
# Define joined_info_function

def classify_index(index_name):
  """Classifies the index name with 'L' for 'l' and 'R' for 'f'."""
  if index_name.startswith('L'):
    return 'L'
  elif index_name.startswith('f'):
    return 'R'
  else:
    # Handle cases where the index doesn't start with 'l' or 'f' (optional)
    return None  # Or any other default value

# Add a column 'side' for the hemisphere of shape
joined_info_shape['side'] = joined_info_shape.index.map(classify_index)

# Define a function to determine 'contraAmpSide', the contralateral side of the missing hand
def get_contra_amp_side(amp_side):
    if amp_side == 'L':
        return 'R'
    elif amp_side == 'R':
        return 'L'
    else:
        return 'none'
# Apply the function to create contraAmpSide column for the contralateral side of the missing hand
joined_info_shape['contraAmpSide'] = joined_info_shape['AmpSide'].apply(get_contra_amp_side)

# Add a study column default to 1
study_column = pd.Series(1, index=joined_info_shape.index)
joined_info_shape['study'] = study_column

# Rename the shape columns
column_mapping = {'1': 'iso1', '2': 'iso2', '3': 'iso3'}
joined_info_shape.rename(columns=column_mapping, inplace=True)

# Define joined_info_function
desired_columns = ['side','contraAmpSide','iso1', 'iso2', 'iso3','UMAP1_U1','UMAP1_U2','UMAP1_U3','UMAP2_U3','UMAP1_U4','UMAP2_U4'] 
# IF needing additional columns
#desired_columns = ['side','contraAmpSide','iso1', 'iso2', 'iso3','UMAP1_U1','UMAP1_U2','UMAP1_U3','UMAP2_U3','UMAP1_U4','UMAP2_U4','UMAP3_U4'] 
joined_info_function = function_info_LR_combined.merge(joined_info_shape[desired_columns],how='inner',left_index=True, right_index=True)

#print('*** function_info_LR_combined ***')
#print(function_info_LR_combined.columns)
#print(len(function_info_LR_combined))
print('*** joined_info_shape ***')
print(len(joined_info_shape))
print(joined_info_shape.columns)
print('*** joined_info_function ***')
print(len(joined_info_function))
print(joined_info_function.columns)

*** joined_info_shape ***
130
Index(['Gender', 'AgeScan', 'AgeLimbLoss', 'Group', 'AmpSide', 'DominantHand',
       'index_ori', 'iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2', 'UMAP1_U3',
       'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4', 'side', 'contraAmpSide', 'study'],
      dtype='object')
*** joined_info_function ***
48
Index(['study', 'gender', 'amp.side', 'birthyear', 'Prosthesisusage',
       'Stumpusage', 'amputatio level', 'Cos', 'Fun', 'Myo', 'missinghand',
       'intacthand', 'residualarm', 'intactarm', 'lips', 'feet', 'side',
       'contraAmpSide', 'iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2',
       'UMAP1_U3', 'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4'],
      dtype='object')


In [28]:
# For anatomical analysis: select amptutee data, contra
ampContraL = joined_info_shape[
    (joined_info_shape['Group'] == 'AMP') &
    (joined_info_shape['contraAmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'L')
]
ampContraR = joined_info_shape[
    (joined_info_shape['Group'] == 'AMP') &
    (joined_info_shape['contraAmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'R')
]
ampContra = pd.concat([ampContraL, ampContraR], axis=0)
# select amputee data, ipsi
ampIpsiL = joined_info_shape[
    (joined_info_shape['Group'] == 'AMP') &
    (joined_info_shape['AmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'L')
]
ampIpsiR = joined_info_shape[
    (joined_info_shape['Group'] == 'AMP') &
    (joined_info_shape['AmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'R')
]
ampIpsi = pd.concat([ampIpsiL, ampIpsiR], axis=0)
# select congenital data, contra
congContraL = joined_info_shape[
    (joined_info_shape['Group'] == 'CONG') &
    (joined_info_shape['contraAmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'L')
]
congContraR = joined_info_shape[
    (joined_info_shape['Group'] == 'CONG') &
    (joined_info_shape['contraAmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'R')
]
congContra = pd.concat([congContraL, congContraR], axis=0)
# select congenital data, ipsi
congIpsiL = joined_info_shape[
    (joined_info_shape['Group'] == 'CONG') &
    (joined_info_shape['AmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'L')
]
congIpsiR = joined_info_shape[
    (joined_info_shape['Group'] == 'CONG') &
    (joined_info_shape['AmpSide'] == joined_info_shape['side']) &
    (joined_info_shape['side'] == 'R')
]
congIpsi = pd.concat([congIpsiL, congIpsiR], axis=0)
# select control data
ctrL = joined_info_shape[
    (joined_info_shape['Group'] == 'CTR') &
    (joined_info_shape['side'] == 'L')
]
ctrR = joined_info_shape[
    (joined_info_shape['Group'] == 'CTR') &
    (joined_info_shape['side'] == 'R')
]
ctr = pd.concat([ctrL,ctrR],axis=0)
# amp and cong group
amp = joined_info_shape[(joined_info_shape['Group'] == 'AMP')]
cong = joined_info_shape[(joined_info_shape['Group'] == 'CONG')]
# contra and ipsi of missing hand group
contra = pd.concat([congContra, ampContra, ctr], axis=0)
ipsi = pd.concat([congIpsi, ampIpsi, ctr], axis=0)

# contraCtrL and contraCtrR
contraCtrL = pd.concat([congContraL, ampContraL, ctrL], axis=0)
contraCtrR = pd.concat([congContraR, ampContraR, ctrR], axis=0)
# ipsiCtrL and ipsiCtrR
ipsiCtrL = pd.concat([congIpsiL, ampIpsiL, ctrL], axis=0)
ipsiCtrR = pd.concat([congIpsiR, ampIpsiR, ctrR], axis=0)

print('congContra:')
print(len(congContra))
print('ampContra:')
print(len(ampContra))
print('amp:')
print(len(amp))
#print(amp.index)
print('cong')
print(len(cong))
#print(cong.index)
print('ctr')
print(len(ctr))
#print(ctr.columns)
print('contra')
print(len(contra))
print('ipsi')
print(len(ipsi))
#print(ipsi.index)
#print(congContra.index_ori)

congContra:
25
ampContra:
16
amp:
32
cong
50
ctr
48
contra
89
ipsi
89


In [30]:
# For activation analysis: select congenital data, contra
congContraL_function = joined_info_function[
    (joined_info_function['contraAmpSide'] == joined_info_function['side']) &
    (joined_info_function['side'] == 'L')
]
congContraR_function = joined_info_function[
    (joined_info_function['contraAmpSide'] == joined_info_function['side']) &
    (joined_info_function['side'] == 'R')
]
congContra_function = pd.concat([congContraL_function, congContraR_function], axis=0)
# select congenital data, ipsi
congIpsiL_function = joined_info_function[
    (joined_info_function['amp.side'] == 0) & (joined_info_function['side'] == 'L')
]
congIpsiR_function = joined_info_function[
    (joined_info_function['amp.side'] == 1) & (joined_info_function['side'] == 'R')
]
congIpsi_function = pd.concat([congIpsiL_function, congIpsiR_function], axis=0)

print('congContra_function:')
print(len(congContra_function))
#print(congContra_function.columns)
print('congIpsi_function:')
print(len(congIpsi_function))
print(congIpsi_function.columns)


congContra_function:
24
congIpsi_function:
24
Index(['study', 'gender', 'amp.side', 'birthyear', 'Prosthesisusage',
       'Stumpusage', 'amputatio level', 'Cos', 'Fun', 'Myo', 'missinghand',
       'intacthand', 'residualarm', 'intactarm', 'lips', 'feet', 'side',
       'contraAmpSide', 'iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2',
       'UMAP1_U3', 'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4'],
      dtype='object')


In [490]:
import os
import shutil
# write meshes to corresponding directories for visualization
#source_dir = r'D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\mesh_isomapCmdsCSSylk5d3_1_distmin_space500_keepOut\'
source_dir = r'D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\mesh_isomapCmdsCSSylk5d3_1_distmin_space500_keepOut'
root_destination_dir = r'D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups'
if not os.path.exists(root_destination_dir):
    os.makedirs(root_destination_dir)
    
prefix = 'coord_500_'
postfix = '.mesh'
# Create new list with prefix and postfix added to index_ori
congContra_mesh_list = congContra['index_ori'].apply(lambda x: f"{prefix}{x}{postfix}").tolist()
congIpsi_mesh_list = congIpsi['index_ori'].apply(lambda x: f"{prefix}{x}{postfix}").tolist()
ampContra_mesh_list = ampContra['index_ori'].apply(lambda x: f"{prefix}{x}{postfix}").tolist()
ampIpsi_mesh_list = ampIpsi['index_ori'].apply(lambda x: f"{prefix}{x}{postfix}").tolist()
ctr_mesh_list = ctr['index_ori'].apply(lambda x: f"{prefix}{x}{postfix}").tolist()

# copy meshes 
destination_dir = root_destination_dir + os.sep + 'ctr'  # Change here !!!
mesh_list = ctr_mesh_list               # Change here !!!
print(destination_dir)
if not os.path.exists(destination_dir):
    os.makedirs(destination_dir)
for file_name in mesh_list:
    src_file_path = os.path.join(source_dir, file_name)
    dst_file_path = os.path.join(destination_dir, file_name)
    
    # Check if the file exists in the source directory before copying
    if os.path.exists(src_file_path):
        shutil.copy(src_file_path, dst_file_path)
        print(f'Copied {file_name} to {destination_dir}')
    else:
        print(f'File {file_name} not found in {source_dir}')
        

D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC01_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC02_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC03_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC04_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC05_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC06_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC07_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC08_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC09_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC10_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_groups\ctr
Copied coord_500_LPC11_struct.mesh to D:\A_joyCODE\DEV_LIN\ampExt\CSSyl\Sub_group

In [42]:
# Shapiro-Wilk test for normality, not normally distributed if p-val < 0.05
from scipy.stats import shapiro

curMeasure = 'UMAP1_U2'

print('congContra')
stat, p_value = shapiro(congContra[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
print('congIpsi')
stat, p_value = shapiro(congIpsi[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
print()
print('ampContra')
stat, p_value = shapiro(ampContra[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
print('ampIpsi')
stat, p_value = shapiro(ampIpsi[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
print()
print('contra:')
stat, p_value = shapiro(contra[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
print('ipsi:')
stat, p_value = shapiro(ipsi[curMeasure])
print(f'{curMeasure} p-val: {p_value}')
print()
print('control')
stat, p_value = shapiro(ctrL[curMeasure])
print(f'ctrL {curMeasure} p-val: {p_value}')
stat, p_value = shapiro(ctrR[curMeasure])
print(f'ctrR {curMeasure} p-val: {p_value}')
stat, p_value = shapiro(ctr[curMeasure])
print(f'ctr {curMeasure} p-val: {p_value}')


congContra
UMAP1_U2 p-val: 0.41291059166820554
congIpsi
UMAP1_U2 p-val: 0.37436034388022227

ampContra
UMAP1_U2 p-val: 0.5084676079374009
ampIpsi
UMAP1_U2 p-val: 0.7898203321469275

contra:
UMAP1_U2 p-val: 0.017857440558486155
ipsi:
UMAP1_U2 p-val: 0.008450748423720381

control
ctrL UMAP1_U2 p-val: 0.05843382553506442
ctrR UMAP1_U2 p-val: 0.10517765258430734
ctr UMAP1_U2 p-val: 0.02078114836074269


In [36]:
# test for difference of mean between contralateral_congenital and control
curMeasure = 'UMAP1_U1'

all_except_congContra = pd.concat([ctr,amp,congIpsi], axis=0)

print(len(all_except_congContra))

# Perform t-test code
t_statistic, p_value = stats.ttest_ind(congContra[curMeasure], ctr[curMeasure])  # Assuming column names are 0 and 1
print(f"t-statistic: {t_statistic}")
print(f"p-value: {p_value}")

w_statistic, p_value = stats.mannwhitneyu(congContra[curMeasure], ctr[curMeasure])
#w_statistic, p_value = stats.mannwhitneyu(congContra[curMeasure], amp[curMeasure])    
#w_statistic, p_value = stats.mannwhitneyu(congContra[curMeasure], congIpsi[curMeasure])    
#w_statistic, p_value = stats.mannwhitneyu(congContra[curMeasure], all_except_congContra[curMeasure])
print(f"Wilcoxon W statistic: {w_statistic}")
print(f"p-value: {p_value}")

105
t-statistic: 2.4308780855585557
p-value: 0.017589830103020804
Wilcoxon W statistic: 807.0
p-value: 0.016372076095056688


In [38]:
# Multiple factor: ols_ordinary least squares regression, then ANOVA (for Normal distribution)
curMeasure = 'UMAP1_U2'
model = ols(curMeasure + '~ C(Group)', data=contra).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)
print()
model = ols(curMeasure + '~ C(Group) + C(side)', data=contra).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)
print()
model = ols(curMeasure + '~ C(Group) + C(side) + C(Gender)', data=joined_info_shape).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)
print()

              sum_sq    df          F    PR(>F)
C(Group)  173.659396   2.0  14.353889  0.000004
Residual  520.232131  86.0        NaN       NaN

              sum_sq    df          F    PR(>F)
C(Group)  164.680521   2.0  13.507838  0.000008
C(side)     2.094263   1.0   0.343562  0.559334
Residual  518.137868  85.0        NaN       NaN

               sum_sq     df         F    PR(>F)
C(Group)    78.604000    2.0  5.413681  0.005561
C(side)     16.276521    1.0  2.242020  0.136826
C(Gender)    0.866983    1.0  0.119423  0.730242
Residual   907.469490  125.0       NaN       NaN



In [40]:
# Multiple factor, Kruskal-Wallis, for non-normal distribution
curMeasure = 'UMAP1_U2'

grouped = contra.groupby(['Group'], observed=False)[curMeasure].apply(list)
stat, p_value = kruskal(*grouped)
print(f'Kruskal-Wallis H-test statistic: {stat}')
print(f'p-value: {p_value}')
print()
grouped = contra.groupby(['Group', 'side'], observed=False)[curMeasure].apply(list)
stat, p_value = kruskal(*grouped)
print(f'Kruskal-Wallis H-test statistic: {stat}')
print(f'p-value: {p_value}')
print()
grouped = contra.groupby(['Group','side','Gender'], observed=False)[curMeasure].apply(list)
stat, p_value = kruskal(*grouped)
print(f'Kruskal-Wallis H-test statistic: {stat}')
print(f'p-value: {p_value}')
print()

Kruskal-Wallis H-test statistic: 22.191121098626752
p-value: 1.5179563379133582e-05

Kruskal-Wallis H-test statistic: 22.684156708875776
p-value: 0.00038787033173058713

Kruskal-Wallis H-test statistic: 23.299313358302186
p-value: 0.016033374375889368



In [468]:
# post-hoc analysis

#!pip install scikit-posthocs
import scikit_posthocs as sp
import warnings

# Suppress the specific FutureWarning
warnings.filterwarnings('ignore', category=FutureWarning, module='scikit_posthocs')
# Create a copy of the dataframe to avoid changing the original
contra_copy = contra.copy()

curMeasure = 'iso1'
# Perform Post-hoc Dunn's test (non-normal distribution)
dunn_result = sp.posthoc_dunn(contra_copy, val_col=curMeasure, group_col='Group', p_adjust='bonferroni')
print('Dunn\'s test result: ' + curMeasure)
print(dunn_result)
print()
curMeasure = 'UMAP1_U2'
# Perform Post-hoc Dunn's test (non-normal distribution)
dunn_result = sp.posthoc_dunn(contra_copy, val_col=curMeasure, group_col='Group', p_adjust='bonferroni')
print('Dunn\'s test result: ' + curMeasure)
print(dunn_result)
print()
curMeasure = 'UMAP2_U4'
# Perform Post-hoc Dunn's test (non-normal distribution)
dunn_result = sp.posthoc_dunn(contra_copy, val_col=curMeasure, group_col='Group', p_adjust='bonferroni')
print('Dunn\'s test result: ' + curMeasure)
print(dunn_result)
print()

Dunn's test result: iso1
           AMP      CONG       CTR
AMP   1.000000  0.102653  1.000000
CONG  0.102653  1.000000  0.017251
CTR   1.000000  0.017251  1.000000

Dunn's test result: UMAP1_U2
           AMP      CONG       CTR
AMP   1.000000  0.001796  1.000000
CONG  0.001796  1.000000  0.000018
CTR   1.000000  0.000018  1.000000

Dunn's test result: UMAP2_U4
           AMP      CONG       CTR
AMP   1.000000  0.010311  1.000000
CONG  0.010311  1.000000  0.000047
CTR   1.000000  0.000047  1.000000



In [469]:
# Activation analysis, correlation: Contralateral
# Pearson for normal distribution, Spearman for non_normal distribution

from scipy.stats import pearsonr, spearmanr

curMeasure = 'UMAP1_U1'
# Pearson's correlation coefficient (for normally distributed data)
pearson_corr, pearson_p_value = pearsonr(congContra_function[curMeasure], congContra_function['missinghand'])
# Spearman's rank correlation coefficient (for non-normally distributed data)
spearman_corr, spearman_p_value = spearmanr(congContra_function[curMeasure], congContra_function['missinghand'])
print('missinghand')
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
pearson_corr, pearson_p_value = pearsonr(congContra_function[curMeasure], congContra_function['intacthand'])
spearman_corr, spearman_p_value = spearmanr(congContra_function[curMeasure], congContra_function['intacthand'])
print('intacthand')
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
pearson_corr, pearson_p_value = pearsonr(congContra_function[curMeasure], congContra_function['residualarm'])
spearman_corr, spearman_p_value = spearmanr(congContra_function[curMeasure], congContra_function['residualarm'])
print('residualarm')
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
pearson_corr, pearson_p_value = pearsonr(congContra_function[curMeasure], congContra_function['intactarm'])
spearman_corr, spearman_p_value = spearmanr(congContra_function[curMeasure], congContra_function['intactarm'])
print('intactarm')
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
pearson_corr, pearson_p_value = pearsonr(congContra_function[curMeasure], congContra_function['lips'])
spearman_corr, spearman_p_value = spearmanr(congContra_function[curMeasure], congContra_function['lips'])
print('lips')
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
pearson_corr, pearson_p_value = pearsonr(congContra_function[curMeasure], congContra_function['feet'])
spearman_corr, spearman_p_value = spearmanr(congContra_function[curMeasure], congContra_function['feet'])
print('feet')
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")

missinghand
Pearson's correlation: -0.18280549251298225, p-value: 0.39255644936210254
Spearman's correlation: -0.19652173913043475, p-value: 0.35737406951590356
intacthand
Pearson's correlation: -0.5279758156009666, p-value: 0.008006004461843207
Spearman's correlation: -0.4765217391304347, p-value: 0.018565115014353994
residualarm
Pearson's correlation: 0.18765591690681094, p-value: 0.3798983070571106
Spearman's correlation: 0.14956521739130432, p-value: 0.48546433225418195
intactarm
Pearson's correlation: -0.4134889374337677, p-value: 0.044595829608145385
Spearman's correlation: -0.4765217391304347, p-value: 0.018565115014353994
lips
Pearson's correlation: -0.5159443580107547, p-value: 0.009857930068608876
Spearman's correlation: -0.4121739130434783, p-value: 0.04534690802018232
feet
Pearson's correlation: -0.40506011350807575, p-value: 0.049583283469618815
Spearman's correlation: -0.29913043478260865, p-value: 0.155619554462795


In [470]:
# Test functional correlation, ipsilateral

curMeasure = 'iso1'
# Pearson's correlation coefficient (for normally distributed data)
pearson_corr, pearson_p_value = pearsonr(congIpsi_function[curMeasure], congIpsi_function['missinghand'])
# Spearman's rank correlation coefficient (for non-normally distributed data)
spearman_corr, spearman_p_value = spearmanr(congIpsi_function[curMeasure], congIpsi_function['missinghand'])
print('missinghand')
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
print('intacthand')
pearson_corr, pearson_p_value = pearsonr(congIpsi_function[curMeasure], congIpsi_function['intacthand'])
spearman_corr, spearman_p_value = spearmanr(congIpsi_function[curMeasure], congIpsi_function['intacthand'])
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
print('residualarm')
pearson_corr, pearson_p_value = pearsonr(congIpsi_function[curMeasure], congIpsi_function['residualarm'])
spearman_corr, spearman_p_value = spearmanr(congIpsi_function[curMeasure], congIpsi_function['residualarm'])
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
print('intactarm')
pearson_corr, pearson_p_value = pearsonr(congIpsi_function[curMeasure], congIpsi_function['intactarm'])
spearman_corr, spearman_p_value = spearmanr(congIpsi_function[curMeasure], congIpsi_function['intactarm'])
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
print('lips')
pearson_corr, pearson_p_value = pearsonr(congIpsi_function[curMeasure], congIpsi_function['lips'])
spearman_corr, spearman_p_value = spearmanr(congIpsi_function[curMeasure], congIpsi_function['lips'])
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")
print('feet')
pearson_corr, pearson_p_value = pearsonr(congIpsi_function[curMeasure], congIpsi_function['feet'])
spearman_corr, spearman_p_value = spearmanr(congIpsi_function[curMeasure], congIpsi_function['feet'])
print(f"Pearson's correlation: {pearson_corr}, p-value: {pearson_p_value}")
print(f"Spearman's correlation: {spearman_corr}, p-value: {spearman_p_value}")

missinghand
Pearson's correlation: 0.234238057715351, p-value: 0.2705990625056293
Spearman's correlation: 0.16695652173913042, p-value: 0.4355381053206143
intacthand
Pearson's correlation: -0.051596462312067104, p-value: 0.8107680162789914
Spearman's correlation: 0.003478260869565217, p-value: 0.9871304734643621
residualarm
Pearson's correlation: -0.08222697990364888, p-value: 0.7024823252485524
Spearman's correlation: -0.12782608695652173, p-value: 0.5516845044816034
intactarm
Pearson's correlation: -0.0520612389716468, p-value: 0.8090939204145127
Spearman's correlation: 0.08956521739130434, p-value: 0.6772689468428972
lips
Pearson's correlation: -0.11104396277458892, p-value: 0.6054608617062391
Spearman's correlation: -0.1652173913043478, p-value: 0.4404029231283094
feet
Pearson's correlation: -0.2514117318050877, p-value: 0.23599250980694608
Spearman's correlation: -0.16695652173913042, p-value: 0.4355381053206143


In [471]:
# Linear model, Multiple Regression, contralateral
import statsmodels.formula.api as smf

curMeasure = 'iso3'

# Encode categorical variables
#congContra_function['study'] = congContra_function['study'].astype('category')

# Define the regression model formula
formula = 'intacthand ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congContra_function).fit()
print('*** intacthand ***')
print(model.summary())
formula = 'missinghand ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congContra_function).fit()
print()
print('*** missinghand ***')
print(model.summary())
formula = 'intactarm ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congContra_function).fit()
print()
print('*** intactarm ***')
print(model.summary())
formula = 'residualarm ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congContra_function).fit()
print()
print('*** residualarm ***')
print(model.summary())
formula = 'lips ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congContra_function).fit()
print()
print('*** lips ***')
print(model.summary())
formula = 'feet ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congContra_function).fit()
print()
print('*** feet ***')
print(model.summary())

*** intacthand ***
                            OLS Regression Results                            
Dep. Variable:             intacthand   R-squared:                       0.225
Model:                            OLS   Adj. R-squared:                  0.109
Method:                 Least Squares   F-statistic:                     1.934
Date:                Mon, 01 Jul 2024   Prob (F-statistic):              0.157
Time:                        20:54:18   Log-Likelihood:                 9.5731
No. Observations:                  24   AIC:                            -11.15
Df Residuals:                      20   BIC:                            -6.434
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.4127      0.127 

In [472]:
# Linear model, Multiple Regression, ipsilateral
#import statsmodels.formula.api as smf
curMeasure = 'iso2'

# Define the regression model formula
formula = 'intacthand ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congIpsi_function).fit()
print('*** intacthand ***')
print(model.summary())
formula = 'missinghand ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congIpsi_function).fit()
print()
print('*** missinghand ***')
print(model.summary())
formula = 'intactarm ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congIpsi_function).fit()
print()
print('*** intactarm ***')
print(model.summary())
formula = 'residualarm ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congIpsi_function).fit()
print()
print('*** residualarm ***')
print(model.summary())
formula = 'lips ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congIpsi_function).fit()
print()
print('*** lips ***')
print(model.summary())
formula = 'feet ~ '+ curMeasure + ' + study + side'
model = smf.ols(formula, data=congIpsi_function).fit()
print()
print('*** feet ***')
print(model.summary())

*** intacthand ***
                            OLS Regression Results                            
Dep. Variable:             intacthand   R-squared:                       0.239
Model:                            OLS   Adj. R-squared:                  0.125
Method:                 Least Squares   F-statistic:                     2.093
Date:                Mon, 01 Jul 2024   Prob (F-statistic):              0.133
Time:                        20:54:19   Log-Likelihood:                 9.7929
No. Observations:                  24   AIC:                            -11.59
Df Residuals:                      20   BIC:                            -6.874
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.3195      0.109 

In [473]:
congContra_function.columns

Index(['study', 'gender', 'amp.side', 'birthyear', 'Prosthesisusage',
       'Stumpusage', 'amputatio level', 'Cos', 'Fun', 'Myo', 'missinghand',
       'intacthand', 'residualarm', 'intactarm', 'lips', 'feet', 'side',
       'contraAmpSide', 'iso1', 'iso2', 'iso3', 'UMAP1_U1', 'UMAP1_U2',
       'UMAP1_U3', 'UMAP2_U3', 'UMAP1_U4', 'UMAP2_U4'],
      dtype='object')

In [474]:
# Partial Correlation

#pip install pingouin
import pingouin as pg

curMeasure = 'UMAP2_U4'
congContra_function['side'] = congContra_function['side'].astype('category').cat.codes
congContra_function['study'] = congContra_function['study'].astype('category').cat.codes
partial_corr = pg.partial_corr(data=congContra_function, x=curMeasure, y='intacthand', covar=['side', 'study'])
print(partial_corr)


          n         r           CI95%     p-val
pearson  24 -0.596673  [-0.81, -0.23]  0.003375
